In [ ]:
import yaml
import re
import os
from pathlib import Path


def load_yaml_variables(yaml_path):
    """
    Load variables from a dbt_project.yml file
    
    Args:
        yaml_path (str): Path to the dbt_project.yml file
    
    Returns:
        dict: Dictionary containing variables defined in the YAML file
    """
    try:
        with open(yaml_path, 'r') as file:
            yaml_data = yaml.safe_load(file)
        
        # Extract variables from the YAML file
        # dbt typically stores variables under 'vars' key
        variables = {}
        if 'vars' in yaml_data:
            variables = yaml_data['vars']
        
        # Check for environment-specific variables
        for key, value in yaml_data.items():
            if isinstance(value, dict) and 'vars' in value:
                # Add environment-specific variables with environment name as prefix
                env_vars = {f"{key}_{var_key}": var_value for var_key, var_value in value['vars'].items()}
                variables.update(env_vars)
        
        return variables
    except Exception as e:
        print(f"Error loading YAML file: {e}")
        return {}

def replace_variables_in_sql(sql_content, variables):
    """
    Replace variables in SQL content with their values
    
    Args:
        sql_content (str): SQL content with variables
        variables (dict): Dictionary of variable names and their values
    
    Returns:
        str: SQL content with variables replaced by their values
    """
    # Pattern to match {{var('variable_name')}} or {{ var('variable_name') }}
    pattern = r"{{\s*var\s*\(\s*['\"](.*?)['\"]\s*\)\s*}}"
    
    def replace_match(match):
        var_name = match.group(1)
        if var_name in variables:
            return str(variables[var_name])
        else:
            # Keep the original variable reference if not found
            print(f"Warning: Variable '{var_name}' not found in YAML file")
            return match.group(0)
    
    # Replace variables in the SQL content
    modified_sql = re.sub(pattern, replace_match, sql_content)
    return modified_sql

def process_sql_file(sql_path, variables):
    """
    Process a single SQL file and replace variables
    
    Args:
        sql_path (str): Path to the SQL file
        variables (dict): Dictionary of variable names and their values
    
    Returns:
        str: SQL content with variables replaced by their values
    """
    try:
        with open(sql_path, 'r') as file:
            sql_content = file.read()
        
        modified_sql = replace_variables_in_sql(sql_content, variables)
        return modified_sql
    except Exception as e:
        print(f"Error processing SQL file {sql_path}: {e}")
        return None

def save_modified_sql(sql_path, modified_sql):
    """
    Save the modified SQL content to a new file
    
    Args:
        sql_path (str): Original SQL file path
        modified_sql (str): Modified SQL content
    
    Returns:
        str: Path to the saved file
    """
    try:
        path = Path(sql_path)
        new_path = path.parent / f"{path.stem}_modified{path.suffix}"
        
        with open(new_path, 'w') as file:
            file.write(modified_sql)
        
        return str(new_path)
    except Exception as e:
        print(f"Error saving modified SQL: {e}")
        return None

def process_directory(dir_path, variables, extension='.sql'):
    """
    Process all SQL files in a directory
    
    Args:
        dir_path (str): Directory path containing SQL files
        variables (dict): Dictionary of variable names and their values
        extension (str, optional): File extension to look for. Defaults to '.sql'.
    
    Returns:
        dict: Dictionary mapping original file paths to modified file paths
    """
    results = {}
    dir_modified_sql = Path('C:/Users/mandald/OneDrive - tata/gitFolder/poc/marketing-data-analytics/test_scripts/output/')
    
    for root, _, files in os.walk(dir_path):
        for file in files:
            if file.endswith(extension):
                sql_path = os.path.join(root, file)
                modified_sql = process_sql_file(sql_path, variables)
                
                if modified_sql:
                    # new_path = save_modified_sql(sql_path, modified_sql)
                    new_path = save_modified_sql(dir_modified_sql.joinpath(file), modified_sql)
                    results[sql_path] = new_path
    
    return results



In [ ]:
# Example usage in a Jupyter notebook

# Load variables from dbt_project.yml
root_dir = Path('C:/Users/mandald/OneDrive - tata/gitFolder/marketing-data-analytics/dags/dbt/mars_dbt/')
# print(root_dir)
yaml_path = root_dir.joinpath('dbt_project.yml')
variables = load_yaml_variables(yaml_path)

# # Print the variables found
# print("Variables found in YAML file:")
# for var_name, var_value in variables.items():
#     print(f"  {var_name}: {var_value}")

# Option 1: Process a single SQL file
sql_path = root_dir.joinpath('models','account_metadata_update','account_metadata.sql')
modified_sql = process_sql_file(sql_path, variables)

if modified_sql:
    # new_path = save_modified_sql(sql_path, modified_sql)
    # print(f"Modified SQL saved to: {new_path}")
    
    # Display the modified SQL
    print("\nModified SQL content:")
    print(modified_sql)

# # Option 2: Process all SQL files in a directory
# sql_dir = root_dir.joinpath('models','account_metadata_update')
# results = process_directory(sql_dir, variables)
# print(f"\nProcessed {len(results)} SQL files")
# for original, modified in results.items():
#     print(f"  {original} -> {modified}")